In [1]:
import scipy.io
import tqdm
MAT_PATH     = '/kaggle/input/datasets/gautam2411/dreamer2/DREAMER.mat'
mat = scipy.io.loadmat(MAT_PATH, simplify_cells=False)
dreamer = mat['DREAMER'][0, 0]
data_all = dreamer['Data'][0]

print(f'Scores keys: {dreamer.dtype.names}')
print(f'Data[0] shape: {data_all.shape}  subjects')
print(f'Data[0] keys: {data_all[0].dtype.names}')

# Inspect first subject
s0     = data_all[0]
eeg0   = s0['EEG'][0, 0]
print(f'EEG keys: {eeg0.dtype.names}')
print(f'stimuli shape: {eeg0["stimuli"][0, 0].shape}  (videos,)')
print(f'stimuli[0,0] shape: {eeg0["stimuli"][0, 0][0, 0].shape}  (samples x channels)')
print(f'baseline[0,0] shape: {eeg0["baseline"][0, 0][0, 0].shape}  (samples x channels)')
print(f'ScoreArousal shape: {s0["ScoreArousal"][0, 0].shape}  (videos x 1)')


Scores keys: ('Data', 'EEG_SamplingRate', 'ECG_SamplingRate', 'EEG_Electrodes', 'noOfSubjects', 'noOfVideoSequences', 'Disclaimer', 'Provider', 'Version', 'Acknowledgement')
Data[0] shape: (23,)  subjects
Data[0] keys: ('Age', 'Gender', 'EEG', 'ECG', 'ScoreValence', 'ScoreArousal', 'ScoreDominance')
EEG keys: ('baseline', 'stimuli')
stimuli shape: (18, 1)  (videos,)
stimuli[0,0] shape: (25472, 14)  (samples x channels)
baseline[0,0] shape: (7808, 14)  (samples x channels)
ScoreArousal shape: (18, 1)  (videos x 1)


In [3]:
import scipy.io
from scipy.signal import welch, butter, filtfilt
from tqdm import tqdm
import numpy as np

def prefilter_trial(eeg):

    """
    eeg : (T,14)

    Returns
    -------
    dict
        filtered EEG for each frequency band
    """

    filtered = {}

    for band, (lo, hi) in EEG_BANDS.items():

        band_sig = np.zeros_like(eeg, dtype=np.float32)

        for ch in range(NUM_CHANS):

            band_sig[:, ch] = bandpass_filter(
                eeg[:, ch].astype(np.float64),
                lo,
                hi
            )

        filtered[band] = band_sig

    return filtered

def bandpass_filter(signal, lowcut, highcut, fs=128, order=4):
    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype="band")

    return filtfilt(b, a, signal)
def window_features(raw_window, filtered_windows):

    """
    raw_window : (64,14)

    filtered_windows :
        dict of 5 filtered windows
    """

    feat = np.zeros((NUM_CHANS,10),dtype=np.float32)

    for ch in range(NUM_CHANS):
        
        #freqs, psd = welch(sig,fs=FS,nperseg=len(sig))
        freqs, psd = welch(
            raw_window.T,
            fs=FS,
            axis=-1,
            nperseg=WIN_SAMP,
            noverlap=0
        )

        psd_feat=[]
        de_feat=[]

        for band,(lo,hi) in EEG_BANDS.items():

            #mask=(freqs>=lo)&(freqs<hi)
            mask = BAND_MASKS[band]
            if np.any(mask):
                power = np.trapezoid(
                    psd[ch, mask],
                    freqs[mask]
                )
            else:
                power = 0.0
           
            psd_feat.append(power)

            filt = filtered_windows[band][:,ch]

            var=max(np.var(filt),1e-10)

            de_feat.append(
                0.5*np.log2(2*np.pi*np.e*var)   # paper Eq.1 uses log2
            )

        feat[ch,:5]=psd_feat
        feat[ch,5:]=de_feat

    return feat


N_SUBJECTS = 23
N_VIDEOS   = 18
NUM_CHANS  = 14
FS= 128
WIN_SEC=0.5
WIN_SAMP   = int(FS * WIN_SEC)   # 64 samples per window

EEG_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}
dummy = np.zeros(WIN_SAMP)

FREQS, _ = welch(
    dummy,
    fs=FS,
    nperseg=WIN_SAMP,
    noverlap=0
)

BAND_MASKS = {
    band: (FREQS >= lo) & (FREQS < hi)
    for band, (lo, hi) in EEG_BANDS.items()
}

all_feats  = []
all_labels = []
all_subj   = []
all_trial  = []

print("Extracting PSD + DE features (per-video baseline correction)...")

for subj_idx in tqdm(range(N_SUBJECTS), desc="Subjects"):

    subj_data = data_all[subj_idx]

    eeg_data = subj_data["EEG"][0, 0]

    scores_ar = subj_data["ScoreArousal"][0, 0].flatten()

    stimuli_cell = eeg_data["stimuli"]
    baseline_cell = eeg_data["baseline"]

    # ---------------------------------------------------
    # Helper to extract one EEG recording
    # ---------------------------------------------------
    def get_video_eeg(field_cell, vid_idx):

        arr = field_cell[0, 0]
    
        if isinstance(arr, np.ndarray) and arr.dtype == object:
            arr = arr.flat[vid_idx]
    
        if arr.shape[0] == NUM_CHANS:
            arr = arr.T
    
        return np.asarray(arr, dtype=np.float32)


    # ===================================================
    # Loop over every video
    # ===================================================
    for vid_idx in range(N_VIDEOS):

        # ------------------------------------------------
        # Baseline EEG of THIS video
        # ------------------------------------------------
        bl_eeg = get_video_eeg(baseline_cell, vid_idx)
        bl_filtered = prefilter_trial(bl_eeg)

        T_bl = bl_eeg.shape[0]
        baseline_features = []

        for start in range(0, T_bl-WIN_SAMP+1, WIN_SAMP):

       
            raw_window = bl_eeg[start:start+WIN_SAMP,:]

            filtered_window = {
                band: bl_filtered[band][start:start+WIN_SAMP]
                for band in EEG_BANDS
            }

            
            feat = window_features(
                raw_window,
                filtered_window
            )

            baseline_features.append(feat)

        baseline_features = np.stack(baseline_features)

        baseline_mean = baseline_features.mean(axis=0)

        # ------------------------------------------------
        # Emotion EEG of SAME video
        # ------------------------------------------------
        emo_eeg = get_video_eeg(stimuli_cell, vid_idx)
        
        emo_filtered = prefilter_trial(emo_eeg)
        
        T_emo = emo_eeg.shape[0]

        label = float(scores_ar[vid_idx])

        for start in range(0, T_emo-WIN_SAMP+1, WIN_SAMP):

            raw_window = emo_eeg[start:start+WIN_SAMP,:]
          
            filtered_window = {
                band: emo_filtered[band][start:start+WIN_SAMP]
                for band in EEG_BANDS
            }
            feat = window_features(
                raw_window,
                filtered_window
            )
            

            corrected = feat - baseline_mean

            all_feats.append(corrected)

            all_labels.append(label)

            all_subj.append(subj_idx)

            all_trial.append(vid_idx)

Extracting PSD + DE features (per-video baseline correction)...


Subjects: 100%|██████████| 23/23 [34:05<00:00, 88.92s/it]


In [4]:
corrected.shape, corrected[0]

((14, 10),
 array([  0.        ,  -4.6267786 , -16.210934  ,  -5.779757  ,
         -0.3922739 ,  -1.0879183 ,  -0.645205  ,  -1.1753035 ,
          0.15147114,  -0.12506843], dtype=float32))

In [6]:
X_raw = np.stack(all_feats).astype(np.float32)

Y = np.array(all_labels, dtype=np.float32)

subject_ids = np.array(all_subj, dtype=np.int32)

trial_ids = np.array(all_trial, dtype=np.int32)

print("X_raw:", X_raw.shape)
print("Y:", Y.shape)
print("Subjects:", subject_ids.shape)
print("Trials:", trial_ids.shape)

import os

SAVE_DIR = "./dreamer_features"

os.makedirs(SAVE_DIR, exist_ok=True)

np.save(os.path.join(SAVE_DIR, "X_raw.npy"), X_raw)

np.save(os.path.join(SAVE_DIR, "Y.npy"), Y)

np.save(os.path.join(SAVE_DIR, "subject_ids.npy"), subject_ids)

np.save(os.path.join(SAVE_DIR, "trial_ids.npy"), trial_ids)

X_raw: (171488, 14, 10)
Y: (171488,)
Subjects: (171488,)
Trials: (171488,)


In [7]:
import os
import copy
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class FusionDataset(Dataset):

    def __init__(self, X):

        self.X = torch.from_numpy(X.astype(np.float32))

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx]

class FusionAutoencoder(nn.Module):

    def __init__(
            self,
            input_dim=140,
            latent_dim=64,
            noise_std=0.2):

        super().__init__()

        self.noise_std = noise_std

        self.encoder = nn.Sequential(

            nn.Linear(input_dim,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64,latent_dim)
        )

        self.decoder = nn.Sequential(

            nn.Linear(latent_dim,64),
            nn.ReLU(),

            nn.Linear(64,128),
            nn.ReLU(),

            nn.Linear(128,input_dim)
        )

    def forward(self,x):

        if self.training:

            x = x + self.noise_std*torch.randn_like(x)

        z = self.encoder(x)

        recon = self.decoder(z)

        return recon,z

def initialize_weights(model):

    for m in model.modules():

        if isinstance(m,nn.Linear):

            nn.init.xavier_uniform_(m.weight)

            nn.init.zeros_(m.bias)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# X_raw comes from the previous cell: shape (N, 14, 10) -> flatten to (N, 140)
X_flat_for_dae = X_raw.reshape(X_raw.shape[0], -1).astype(np.float32)

train_dataset = FusionDataset(X_flat_for_dae)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)

print(f'device       : {device}')
print(f'train_loader : {len(train_dataset)} samples, '
      f'{len(train_loader)} batches of size 256')


device       : cuda
train_loader : 171488 samples, 670 batches of size 256


In [18]:
def weights_init(m):

    if isinstance(m, nn.Linear):

        nn.init.xavier_uniform_(m.weight)

        nn.init.zeros_(m.bias)

class FusionAutoencoder(nn.Module):

    def __init__(self,
                 input_dim=140,
                 latent_dim=64,
                 noise_std=0.2):

        super().__init__()

        self.noise_std = noise_std

        self.encoder = nn.Sequential(

            nn.Linear(input_dim,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64,latent_dim)
        )

        self.decoder = nn.Sequential(

            nn.Linear(latent_dim,64),
            nn.ReLU(),

            nn.Linear(64,128),
            nn.ReLU(),

            nn.Linear(128,input_dim)
        )

    def forward(self,x):

        if self.training:
            x_noisy = x + self.noise_std*torch.randn_like(x)
        else:
            x_noisy = x

        z = self.encoder(x_noisy)

        recon = self.decoder(z)

        return recon,z
        

model=FusionAutoencoder().to(device)
model.apply(weights_init)   # now safe — model exists

criterion=nn.MSELoss()

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

for epoch in range(100):

    model.train()

    loss_epoch=0

    for x in train_loader:

        x=x.to(device)

        recon,z=model(x)

        loss=criterion(recon,x)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        loss_epoch+=loss.item()

    print(epoch,loss_epoch/len(train_loader))


torch.save(model.encoder.state_dict(),
           "fusion_encoder.pt")

class EmotionClassifier(nn.Module):

    def __init__(self,
                 latent_dim=64,
                 n_classes=2):

        super().__init__()

        self.encoder=nn.Sequential(

            nn.Linear(140,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64,latent_dim)
        )

        self.classifier=nn.Sequential(

            nn.Linear(latent_dim,32),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(32,n_classes)
        )

    def forward(self,x):

        z=self.encoder(x)

        out=self.classifier(z)

        return out


model=EmotionClassifier()

model.encoder.load_state_dict(
    torch.load("fusion_encoder.pt")
)

criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam([

    {
        "params":model.encoder.parameters(),
        "lr":1e-5
    },

    {
        "params":model.classifier.parameters(),
        "lr":1e-3
    }

])

0 18442887.2375
1 13387548.635261195
2 10975963.913619403
3 9430708.12164179
4 8589216.326399254
5 7759495.56380597
6 7256585.186287314
7 6796788.389458955
8 7066895.220615672
9 6790898.752052239
10 6532950.460634328
11 5915314.494402985
12 6395342.895755597
13 5838113.364552239
14 6602589.427938432
15 6848993.7817164175
16 6310638.878264925
17 5872239.246968283
18 5769328.842817164
19 7277066.210027985
20 5811906.332369403
21 5553031.754337686
22 6698682.761567164
23 5833546.764972015
24 5694267.164645523
25 5617265.03959888
26 5853337.689598881
27 5869183.858488806
28 5484232.5047108205
29 6180487.739785448
30 5572684.185914179
31 5682993.068516791
32 5249442.152658582
33 5174946.708255597
34 5145602.063479478
35 5356611.353264925
36 6272187.533069029
37 5022985.9237873135
38 4838025.250419776
39 5183181.778591418
40 4954516.417770523
41 4979232.172014926
42 5016640.435914179
43 5155465.3859141795
44 4858869.87472015
45 5425392.647527985
46 4941059.233302238
47 5084975.22709888
48 45

---
## Config — ALL constants needed from here to the end of the notebook

**This cell must never be deleted or skipped.** Every fix applied to this project so far has been lost multiple times because this cell went missing between uploads. If you edit anything below, keep this cell.

In [8]:
import math, collections
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy import stats as sp_stats

BASE_DIR     = SAVE_DIR                                   # from Cell 3
X_RAW_PATH   = os.path.join(BASE_DIR, "X_raw.npy")
Y_RAW_PATH   = os.path.join(BASE_DIR, "Y.npy")
SUBJ_PATH    = os.path.join(BASE_DIR, "subject_ids.npy")
TRIAL_PATH   = os.path.join(BASE_DIR, "trial_ids.npy")    # needed for video-wise LOSO
X_FUSED_PATH = os.path.join(BASE_DIR, "X_fused.npy")

RAW_DIM     = 10                 # [5 PSD, 5 DE] per channel
LATENT_DIM  = 32
IN_CHANNELS = NUM_CHANS          # 14 -- paper: X in R^(c x l), c=channels
SEQ_LEN     = LATENT_DIM         # 32 -- paper: l=latent feature length

# Paper §4.2: "Ratings of 5(3) and above were classified as high" -> >=
# THIS EXACT LINE HAS BEEN REVERTED TO '>' MULTIPLE TIMES ACROSS THIS
# PROJECT AND EACH TIME CAPPED ACCURACY NEAR 55-60%. DO NOT CHANGE IT BACK.
THRESHOLD    = 3.0

SEED         = 42
BATCH_SIZE   = 128
K_FOLDS      = 5
EPOCHS_INDEP = 50

# Paper Table 1
LR           = 1e-4
WEIGHT_DECAY = 0.06

np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f'X_RAW_PATH   : {X_RAW_PATH}')
print(f'TRIAL_PATH   : {TRIAL_PATH}')
print(f'X_FUSED_PATH : {X_FUSED_PATH}')
print(f'IN_CHANNELS={IN_CHANNELS}  SEQ_LEN={SEQ_LEN}  THRESHOLD={THRESHOLD}')


X_RAW_PATH   : ./dreamer_features/X_raw.npy
TRIAL_PATH   : ./dreamer_features/trial_ids.npy
X_FUSED_PATH : ./dreamer_features/X_fused.npy
IN_CHANNELS=14  SEQ_LEN=32  THRESHOLD=3.0


---
## Denoising Autoencoder (DAE)


In [9]:
class FeatureFusionDAE(nn.Module):
    def __init__(self, input_dim=RAW_DIM, hidden_dim=64,
                 latent_dim=LATENT_DIM, noise_std=0.20):
        super().__init__()
        self.noise_std = noise_std
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(hidden_dim, latent_dim),
            nn.BatchNorm1d(latent_dim),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        nx  = x + self.noise_std * torch.randn_like(x) if self.training else x
        lat = self.encoder(nx)
        return self.decoder(lat), lat


X_raw    = np.load(X_RAW_PATH)   # (N, 14, 10)
subj_ids = np.load(SUBJ_PATH)    # (N,)

# Z-score on flattened features (global, for DAE training only)
X_flat = X_raw.reshape(-1, RAW_DIM).astype(np.float32)
mu = X_flat.mean(0, keepdims=True)
sd = X_flat.std(0,  keepdims=True); sd[sd < 1e-8] = 1.0
X_norm = (X_flat - mu) / sd
print(f'Z-score: mean={X_norm.mean():.4f}  std={X_norm.std():.4f}')

dae      = FeatureFusionDAE().to(device)
dae_opt  = optim.Adam(dae.parameters(), lr=1e-3, weight_decay=1e-4)
dae_crit = nn.MSELoss()
dae_ld   = DataLoader(TensorDataset(torch.from_numpy(X_norm)),
                      batch_size=1024, shuffle=True)

print(f'Training DAE: {RAW_DIM}->64->{LATENT_DIM} | 100 epochs')
for ep in range(1, 101):
    dae.train(); ls = 0
    for (bx,) in dae_ld:
        bx = bx.to(device); rc, _ = dae(bx)
        l = dae_crit(rc, bx)
        dae_opt.zero_grad(); l.backward(); dae_opt.step()
        ls += l.item()
    if ep % 10 == 0 or ep == 1:
        print(f'  Ep {ep:>3}/100  Loss: {ls/len(dae_ld):.6f}')

# Encode all samples
dae.eval(); lats = []
with torch.no_grad():
    for (bx,) in DataLoader(TensorDataset(torch.from_numpy(X_norm)),
                             1024, shuffle=False):
        _, lat = dae(bx.to(device))
        lats.append(lat.cpu().numpy())

X_fused_flat = np.concatenate(lats)                          # (N*14, 32)
X_fused      = X_fused_flat.reshape(X_raw.shape[0], NUM_CHANS, LATENT_DIM)
np.save(X_FUSED_PATH, X_fused)
print(f'X_fused: {X_fused.shape}  saved')

Z-score: mean=-0.0000  std=0.9496
Training DAE: 10->64->32 | 100 epochs
  Ep   1/100  Loss: 0.062150
  Ep  10/100  Loss: 0.026171
  Ep  20/100  Loss: 0.025126
  Ep  30/100  Loss: 0.024329
  Ep  40/100  Loss: 0.024099
  Ep  50/100  Loss: 0.023605
  Ep  60/100  Loss: 0.023365
  Ep  70/100  Loss: 0.023353
  Ep  80/100  Loss: 0.023116
  Ep  90/100  Loss: 0.023174
  Ep 100/100  Loss: 0.023181
X_fused: (171488, 14, 32)  saved


---
## Prepare Tensors


In [10]:
X_fused   = np.load(X_FUSED_PATH)   # (N, 14, 32)
Y_raw     = np.load(Y_RAW_PATH)     # (N,)
subj_ids  = np.load(SUBJ_PATH)      # (N,)
trial_ids = np.load(TRIAL_PATH)     # (N,)

Y_binary = (Y_raw > THRESHOLD).astype(np.int64)

min_per_video = {}
for vid in range(N_VIDEOS):
    counts = [((subj_ids == sid) & (trial_ids == vid)).sum()
              for sid in range(N_SUBJECTS)]
    counts = [c for c in counts if c > 0]
    min_per_video[vid] = min(counts) if counts else 0

print('Per-video window cap (min across all subjects):')
for vid in range(N_VIDEOS):
    print(f'  Video {vid+1:>2}: {min_per_video[vid]} windows/subject')

rng = np.random.default_rng(SEED)
strat_idx = []
for sid in range(N_SUBJECTS):
    for vid in range(N_VIDEOS):
        m   = np.where((subj_ids == sid) & (trial_ids == vid))[0]
        cap = min_per_video[vid]
        if len(m) == 0 or cap == 0:
            continue
        strat_idx.append(rng.choice(m, cap, replace=False))

strat_idx = np.concatenate(strat_idx)
rng.shuffle(strat_idx)

X_bal = X_fused[strat_idx]
Y_bal = Y_binary[strat_idx]
S_bal = subj_ids[strat_idx]
T_bal = trial_ids[strat_idx]

X_tensor = torch.FloatTensor(X_bal)
Y_tensor = torch.LongTensor(Y_bal)

print(f'\nTotal windows per subject: {len(X_bal) // N_SUBJECTS}  (identical for every subject)')
print(f'X_tensor: {X_tensor.shape}')
print(f'High={(Y_bal==1).sum()}  Low={(Y_bal==0).sum()}  '
      f'(imbalance now handled via loss weights, not discarding)')

# ── Class weights, since we no longer discard to force 50/50 balance ──
class_counts  = np.bincount(Y_bal, minlength=2)
class_weights = torch.FloatTensor(len(Y_bal) / (2.0 * class_counts)).to(device)
print(f'Class weights for CrossEntropyLoss: {class_weights.tolist()}')

assert X_tensor.shape[1] == IN_CHANNELS and X_tensor.shape[2] == SEQ_LEN, \
    f"Shape mismatch: got {X_tensor.shape}, expected (N,{IN_CHANNELS},{SEQ_LEN})"

Per-video window cap (min across all subjects):
  Video  1: 398 windows/subject
  Video  2: 262 windows/subject
  Video  3: 696 windows/subject
  Video  4: 332 windows/subject
  Video  5: 272 windows/subject
  Video  6: 380 windows/subject
  Video  7: 384 windows/subject
  Video  8: 788 windows/subject
  Video  9: 290 windows/subject
  Video 10: 134 windows/subject
  Video 11: 192 windows/subject
  Video 12: 362 windows/subject
  Video 13: 736 windows/subject
  Video 14: 340 windows/subject
  Video 15: 616 windows/subject
  Video 16: 390 windows/subject
  Video 17: 512 windows/subject
  Video 18: 372 windows/subject

Total windows per subject: 7456  (identical for every subject)
X_tensor: torch.Size([171488, 14, 32])
High=82178  Low=89310  (imbalance now handled via loss weights, not discarding)
Class weights for CrossEntropyLoss: [0.9600716829299927, 1.043393611907959]


---
## MSCBlock

In [11]:
class MSCBlock(nn.Module):
    """
    Multi-scale convolutional block (paper §3.3 Eqs 6-7).
    No internal dropout -- not part of the paper's design.
    """
    def __init__(self, in_ch, mid_ch=32):
        super().__init__()
        self.t4  = nn.Conv2d(in_ch,    mid_ch, (1, 4),  padding='same')
        self.t8  = nn.Conv2d(in_ch,    mid_ch, (1, 8),  padding='same')
        self.t16 = nn.Conv2d(in_ch,    mid_ch, (1, 16), padding='same')
        self.tn  = nn.BatchNorm2d(mid_ch * 3)

        self.c4  = nn.Conv2d(mid_ch*3, mid_ch, (4, 1), padding='same')
        self.c2  = nn.Conv2d(mid_ch*3, mid_ch, (2, 1), padding='same')
        self.cn  = nn.BatchNorm2d(mid_ch * 2)

        self.proj     = nn.Conv2d(mid_ch*2, in_ch, 1)
        self.out_norm = nn.BatchNorm2d(in_ch)
        self.act      = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        t = self.tn(torch.cat([
            self.act(self.t4(x)),
            self.act(self.t8(x)),
            self.act(self.t16(x)),
        ], dim=1))
        c = self.cn(torch.cat([
            self.act(self.c4(t)),
            self.act(self.c2(t)),
        ], dim=1))
        return self.out_norm(self.proj(c)) + x


---
## MSC-TimesNet Full Model

In [12]:
 
class MSCTimesNet(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, in_channels=IN_CHANNELS,
                 top_k=3, mid_ch=32, hidden_dim=256,
                 num_classes=2, dropout=0.25):
        super().__init__()
        self.seq_len = seq_len
        self.in_channels = in_channels
        self.top_k = top_k
        self.msc_block = MSCBlock(in_ch=in_channels, mid_ch=mid_ch)
        flat_dim = seq_len * in_channels    # 448

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(flat_dim),
            nn.Linear(flat_dim, hidden_dim),
            nn.LeakyReLU(0.2, inplace=True),    
            nn.Dropout(dropout),                

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2, inplace=True),    
            nn.Dropout(dropout / 2),

            nn.Linear(hidden_dim // 2, num_classes),
        )

    def _to_2d(self, x, period, freq):
        B, C, L = x.shape
        tgt = period * freq
        x = F.pad(x, (0, tgt - L)) if tgt > L else x[:, :, :tgt]
        return x.reshape(B, C, period, freq)

    def forward(self, x):
        B, C, L = x.shape

        # FFT period detection (paper Eqs 2-4)
        amp = torch.abs(torch.fft.rfft(x, dim=-1)).mean(dim=1)
        amp = amp.clone(); amp[:, 0] = 0.0
        k   = min(self.top_k, amp.shape[-1] - 1)
        top_a, top_f = torch.topk(amp, k, dim=-1)
        top_f = top_f.clamp(min=1)
        w = F.softmax(top_a, dim=-1)       # (B, k)

        res = torch.zeros_like(x)
        for i in range(k):
            fi  = int(top_f[:, i].float().mean().round().clamp(min=1).item())
            pi  = math.ceil(L / fi)
            x2d = self._to_2d(x, pi, fi)           # (B, C, pi, fi)
            x2d = self.msc_block(x2d)              # (B, C, pi, fi) — same size
            x1d = x2d.reshape(B, C, -1)[:, :, :L]  # (B, C, L) — trim to original length
            res = res + w[:, i].view(B, 1, 1) * x1d

        return self.classifier(x + res)


def build_model(num_classes=2, seq_len=SEQ_LEN):
    return MSCTimesNet(
        seq_len=seq_len,
        in_channels=IN_CHANNELS,
        top_k=3,
        mid_ch=32,
        hidden_dim=256,
        num_classes=num_classes,
        dropout=0.25,  # paper spec
    ).to(device)


print(f'Params: {sum(p.numel() for p in build_model().parameters()):,}')
dummy = torch.randn(4, IN_CHANNELS, SEQ_LEN, device=device)
print(f'Test forward: {tuple(build_model()(dummy).shape)}')  # should be (4, 2)

Params: 181,388
Test forward: (4, 2)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(


---
## Training Utilities

In [13]:
def train_epoch(model, loader, criterion, optimizer):
    model.train(); ls = co = tot = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device); optimizer.zero_grad()
        logits = model(xb); loss = criterion(logits, yb); loss.backward()
        optimizer.step()   # no gradient clipping -- paper does not use it
        ls += loss.item(); co += (logits.argmax(1) == yb).sum().item(); tot += yb.size(0)
    return ls / len(loader), co / tot

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ap, al = [], []
    for xb, yb in loader:
        p = model(xb.to(device)).argmax(1).cpu().numpy(); ap.append(p); al.append(yb.numpy())
    p = np.concatenate(ap); l = np.concatenate(al)
    return accuracy_score(l, p), f1_score(l, p, average='macro', zero_division=0), p, l

def make_loaders(X_tr, Y_tr, X_te, Y_te, bs=BATCH_SIZE):
    tr = DataLoader(TensorDataset(X_tr, Y_tr), bs, shuffle=True, drop_last=True)
    te = DataLoader(TensorDataset(X_te, Y_te), bs, shuffle=False)
    return tr, te

---
## Subject-Independent 5-Fold CV + Per-Subject Metrics


In [24]:
gkf = GroupKFold(n_splits=K_FOLDS)
X_np = X_tensor.numpy(); Y_np = Y_tensor.numpy(); G_np = S_bal
indep_rows = []; subj_indep = collections.defaultdict(list)

print(f'Subject-INDEPENDENT {K_FOLDS}-Fold | {EPOCHS_INDEP} epochs/fold')
print('='*70)

for fold, (tr_i, te_i) in enumerate(gkf.split(X_np, Y_np, groups=G_np), 1):
    tr_subs = np.unique(G_np[tr_i]); te_subs = np.unique(G_np[te_i])
    print(f'\n-- Fold {fold}/{K_FOLDS} --')
    print(f'   Train subjects ({len(tr_subs)}): {tr_subs+1}')
    print(f'   Test  subjects ({len(te_subs)}): {te_subs+1}')
    assert len(set(tr_subs) & set(te_subs)) == 0, 'Subject leakage!'

    # Features already z-scored + DAE-encoded -- no re-normalisation
    Xtr = torch.FloatTensor(X_np[tr_i]); Ytr = torch.LongTensor(Y_np[tr_i])
    Xte = torch.FloatTensor(X_np[te_i]); Yte = torch.LongTensor(Y_np[te_i])
    Gte = G_np[te_i]

    tr_ld, te_ld = make_loaders(Xtr, Ytr, Xte, Yte, bs=128)

    model = build_model()
    crit  = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ba = bf = 0; bp = os.path.join(BASE_DIR, f'indep_f{fold}.pth')

    for ep in range(1, EPOCHS_INDEP + 1):
        tl, ta = train_epoch(model, tr_ld, crit, opt)
        ta2, tf, _, _ = evaluate(model, te_ld)
        if tf > bf: ba = ta2; bf = tf; torch.save(model.state_dict(), bp)
        if ep % 10 == 0 or ep == 1:
            print(f'  Ep{ep:>3} Loss:{tl:.4f} Tr:{ta:.4f} '
                  f'TeAcc:{ta2:.4f} F1:{tf:.4f} lr:{opt.param_groups[0]["lr"]:.2e}')

    model.load_state_dict(torch.load(bp, map_location=device))
    fa, ff, fp, fl = evaluate(model, te_ld)
    indep_rows.append({'Fold': fold, 'Subs': te_subs, 'Acc': fa, 'F1': ff})

    print(f'\n  Fold {fold} Best -> Acc:{fa:.4f}  F1:{ff:.4f}')
    print(classification_report(fl, fp, target_names=['Low', 'High'], digits=4))
    cm = confusion_matrix(fl, fp)
    print(f'  Confusion Matrix:')
    print(f'               Pred Low  Pred High')
    print(f'  Actual Low   {cm[0,0]:>8}  {cm[0,1]:>9}')
    print(f'  Actual High  {cm[1,0]:>8}  {cm[1,1]:>9}')

    print(f'  Per-Subject (Fold {fold}):')
    print(f'  {"Subj":<8} {"N":>6} {"Acc":>9} {"F1":>9}')
    print(f'  {"-"*36}')
    for sid in te_subs:
        m = Gte == sid
        if not m.any(): continue
        sa = accuracy_score(fl[m], fp[m]); sf = f1_score(fl[m], fp[m], average='macro', zero_division=0)
        subj_indep[sid].append((sa, sf))
        print(f'  S{sid+1:<7} {m.sum():>6} {sa:>9.4f} {sf:>9.4f}')

ia = [r['Acc'] for r in indep_rows]; if1 = [r['F1'] for r in indep_rows]
print(f'\n{"="*70}')
print('  SUBJECT-INDEPENDENT 5-FOLD RESULTS')
print(f'{"="*70}')
print(f'  {"Fold":<6} {"Test Subjects":<30} {"Acc":>8}  {"F1":>8}')
print(f'  {"-"*56}')
for r in indep_rows:
    print(f'  {r["Fold"]:<6} {str(list(r["Subs"]+1)):<30} {r["Acc"]:>8.4f}  {r["F1"]:>8.4f}')
print(f'  {"-"*56}')
print(f'  Mean  {np.mean(ia):>39.4f}  {np.mean(if1):>8.4f}')
print(f'  Std   {np.std(ia):>39.4f}  {np.std(if1):>8.4f}')

print(f'\n  PER-SUBJECT RESULTS')
print(f'  {"Subj":<8} {"Acc":>9} {"F1":>9}')
print(f'  {"-"*30}')
all_s_accs = []
for sid in sorted(subj_indep):
    accs = [e[0] for e in subj_indep[sid]]; f1s = [e[1] for e in subj_indep[sid]]
    all_s_accs.append(np.mean(accs))
    print(f'  S{sid+1:<7} {np.mean(accs):>9.4f} {np.mean(f1s):>9.4f}')
print(f'  {"-"*30}\n  Overall mean: {np.mean(all_s_accs):.4f}')

Subject-INDEPENDENT 5-Fold | 50 epochs/fold

-- Fold 1/5 --
   Train subjects (19): [ 1  2  3  4  5  6  8  9 10 11 12 15 16 17 18 19 20 21 22]
   Test  subjects (4): [ 7 13 14 23]
  Ep  1 Loss:0.6315 Tr:0.6321 TeAcc:0.4398 F1:0.4322 lr:1.00e-04
  Ep 10 Loss:0.4920 Tr:0.7431 TeAcc:0.4589 F1:0.4571 lr:1.00e-04
  Ep 20 Loss:0.4491 Tr:0.7735 TeAcc:0.4682 F1:0.4651 lr:1.00e-04
  Ep 30 Loss:0.4195 Tr:0.7912 TeAcc:0.4692 F1:0.4684 lr:1.00e-04
  Ep 40 Loss:0.4002 Tr:0.8046 TeAcc:0.4923 F1:0.4908 lr:1.00e-04
  Ep 50 Loss:0.3862 Tr:0.8126 TeAcc:0.4886 F1:0.4873 lr:1.00e-04

  Fold 1 Best -> Acc:0.5143  F1:0.5107
              precision    recall  f1-score   support

         Low     0.5122    0.6003    0.5527     11838
        High     0.5172    0.4283    0.4686     11838

    accuracy                         0.5143     23676
   macro avg     0.5147    0.5143    0.5107     23676
weighted avg     0.5147    0.5143    0.5107     23676

  Confusion Matrix:
               Pred Low  Pred High
  Actual

## Leave-One-Subject-Out (LOSO) — Per-Subject AND Per-Video Results


In [14]:
_required = ['X_tensor', 'Y_tensor', 'S_bal', 'T_bal']
_missing  = [v for v in _required if v not in dir()]
if _missing:
    raise RuntimeError(
        f"Missing variable(s): {_missing}\n"
        f"FIX: scroll up and re-run the 'Prepare Tensors' cell "
        f"(the one that creates X_tensor, Y_tensor, S_bal, T_bal) "
        f"before running this LOSO cell."
    )

EPOCHS_LOSO = 50   # reduced from 150 for 23-fold runtime; raise for final numbers

X_np = X_tensor.numpy(); Y_np = Y_tensor.numpy()
G_np = S_bal    # subject id per balanced sample
V_np = T_bal    # video (trial) id per balanced sample

loso_subject_rows = []                       # per-subject summary
loso_video_rows   = collections.defaultdict(list)   # subj -> list of video-level dicts

print(f'True LOSO: {N_SUBJECTS} folds | {EPOCHS_LOSO} epochs/fold')
print('='*78)

for held_out in range(N_SUBJECTS):
    tr_mask = G_np != held_out
    te_mask = G_np == held_out
    if te_mask.sum() == 0:
        print(f'-- Subject {held_out+1}: no samples in balanced set, skipped --')
        continue

    Xtr = torch.FloatTensor(X_np[tr_mask]); Ytr = torch.LongTensor(Y_np[tr_mask])
    Xte = torch.FloatTensor(X_np[te_mask]); Yte = torch.LongTensor(Y_np[te_mask])
    Vte = V_np[te_mask]     # video id for every test window

    tr_ld, te_ld = make_loaders(Xtr, Ytr, Xte, Yte, bs=128)

    model = build_model()
    crit = nn.CrossEntropyLoss(weight=class_weights)
    opt   = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    bp    = os.path.join(BASE_DIR, f'loso_s{held_out+1}.pth')
    best_f1 = 0

    for ep in range(1, EPOCHS_LOSO + 1):
        tl, ta = train_epoch(model, tr_ld, crit, opt)
        ta2, tf, _, _ = evaluate(model, te_ld)
        if tf > best_f1:
            best_f1 = tf
            torch.save(model.state_dict(), bp)

    model.load_state_dict(torch.load(bp, map_location=device))
    s_acc, s_f1, s_pred, s_true = evaluate(model, te_ld)

    loso_subject_rows.append({'Subject': held_out+1, 'N': te_mask.sum(),
                               'Acc': s_acc, 'F1': s_f1})

    print(f'\n{"="*78}')
    print(f'  SUBJECT {held_out+1}  (N={te_mask.sum()} windows)')
    print(f'{"="*78}')
    print(f'  Subject-level  Acc: {s_acc:.4f}   F1: {s_f1:.4f}')
    cm = confusion_matrix(s_true, s_pred, labels=[0, 1])
    print(f'  Confusion Matrix:')
    print(f'               Pred Low  Pred High')
    print(f'  Actual Low   {cm[0,0]:>8}  {cm[0,1]:>9}')
    print(f'  Actual High  {cm[1,0]:>8}  {cm[1,1]:>9}')

    # ── Video-wise breakdown ────────────────────────────────────────────────
    print(f'\n  Video-wise results:')
    print(f'  {"Video":>5} {"N win":>6} {"WinAcc":>8} {"WinF1":>8} '
          f'{"MajVote":>8} {"TrueLbl":>8} {"Match":>6}')
    print(f'  {"-"*54}')

    for vid in sorted(np.unique(Vte)):
        vmask = Vte == vid
        v_pred = s_pred[vmask]
        v_true = s_true[vmask]

        v_acc = accuracy_score(v_true, v_pred)
        v_f1  = f1_score(v_true, v_pred, average='macro', zero_division=0)

        maj_vote  = int(sp_stats.mode(v_pred, keepdims=False).mode)
        true_lbl  = int(v_true[0])          # all windows of one video share the label
        match     = 'Yes' if maj_vote == true_lbl else 'No'

        lbl_str = {0: 'Low', 1: 'High'}
        print(f'  {vid+1:>5} {vmask.sum():>6} {v_acc:>8.4f} {v_f1:>8.4f} '
              f'{lbl_str[maj_vote]:>8} {lbl_str[true_lbl]:>8} {match:>6}')

        loso_video_rows[held_out+1].append({
            'Video': vid+1, 'N': vmask.sum(), 'WinAcc': v_acc, 'WinF1': v_f1,
            'MajVote': maj_vote, 'TrueLbl': true_lbl, 'Match': match == 'Yes',
        })

# ── Grand summary ─────────────────────────────────────────────────────────
print(f'\n{"="*78}')
print('  LOSO GRAND SUMMARY')
print(f'{"="*78}')
print(f'  {"Subject":<9} {"N":>6} {"Acc":>8} {"F1":>8}')
print(f'  {"-"*33}')
for r in loso_subject_rows:
    print(f'  S{r["Subject"]:<8} {r["N"]:>6} {r["Acc"]:>8.4f} {r["F1"]:>8.4f}')

accs = [r['Acc'] for r in loso_subject_rows]
f1s  = [r['F1']  for r in loso_subject_rows]
print(f'  {"-"*33}')
print(f'  Mean       {np.mean(accs):>8.4f} {np.mean(f1s):>8.4f}')
print(f'  Std        {np.std(accs):>8.4f} {np.std(f1s):>8.4f}')

# ── Video-level aggregate accuracy (majority-vote correctness rate) ───────
all_video_matches = [v['Match'] for rows in loso_video_rows.values() for v in rows]
print(f'\n  Video-level majority-vote accuracy (across all {len(all_video_matches)} videos): '
      f'{np.mean(all_video_matches):.4f}')


True LOSO: 23 folds | 50 epochs/fold

  SUBJECT 1  (N=7456 windows)
  Subject-level  Acc: 0.4738   F1: 0.4519
  Confusion Matrix:
               Pred Low  Pred High
  Actual Low       2512       2604
  Actual High      1319       1021

  Video-wise results:
  Video  N win   WinAcc    WinF1  MajVote  TrueLbl  Match
  ------------------------------------------------------
      1    398   0.8668   0.4643      Low      Low    Yes
      2    262   0.5878   0.3702      Low      Low    Yes
      3    696   0.3060   0.2343      Low     High     No
      4    332   0.8735   0.4662      Low      Low    Yes
      5    272   0.7353   0.4237     High     High    Yes
      6    380   0.5316   0.3471      Low      Low    Yes
      7    384   0.5052   0.3356     High     High    Yes
      8    788   0.0178   0.0175     High      Low     No
      9    290   0.0276   0.0268     High      Low     No
     10    134   0.1119   0.1007     High      Low     No
     11    192   0.8646   0.4637      Low      